In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

In [5]:
expression_df = pd.read_csv("TCGA-BRCA.STAR_TPM.tsv.gz", sep='\t', index_col=0)
print("Shape:", expression_df.shape)
print("First few genes:", expression_df.head())

Shape: (60660, 1226)
First few genes:                     TCGA-D8-A146-01A  TCGA-AQ-A0Y5-01A  TCGA-C8-A274-01A  \
Ensembl_ID                                                                 
ENSG00000000003.15          5.662037          3.703721          6.514515   
ENSG00000000005.6           3.376096          0.463099          0.000000   
ENSG00000000419.13          6.860140          7.086452          6.805072   
ENSG00000000457.14          4.400552          4.051007          5.037264   
ENSG00000000460.17          2.845169          2.426989          4.043248   

                    TCGA-BH-A0BD-01A  TCGA-B6-A1KC-01B  TCGA-AC-A62V-01A  \
Ensembl_ID                                                                 
ENSG00000000003.15          4.784917          4.251984          3.536849   
ENSG00000000005.6           2.328061          0.435415          0.736475   
ENSG00000000419.13          6.443071          6.178436          6.823355   
ENSG00000000457.14          4.374970          3.5

In [9]:
# Average TPM per gene across tumor samples
avg_expr = expression_df.mean(axis=1)

# Select top 1000 highly expressed genes
top_genes = avg_expr.sort_values(ascending=False).head(1000).index.tolist()
top_genes


['ENSG00000198938.2',
 'ENSG00000198712.1',
 'ENSG00000198804.2',
 'ENSG00000198886.2',
 'ENSG00000210082.2',
 'ENSG00000198899.2',
 'ENSG00000198727.2',
 'ENSG00000198888.2',
 'ENSG00000198840.2',
 'ENSG00000198763.3',
 'ENSG00000087086.15',
 'ENSG00000034510.6',
 'ENSG00000212907.2',
 'ENSG00000198695.2',
 'ENSG00000184009.12',
 'ENSG00000198786.2',
 'ENSG00000112306.8',
 'ENSG00000211592.8',
 'ENSG00000075624.17',
 'ENSG00000228253.1',
 'ENSG00000231500.7',
 'ENSG00000211459.2',
 'ENSG00000142534.7',
 'ENSG00000163191.6',
 'ENSG00000210196.2',
 'ENSG00000161016.18',
 'ENSG00000177954.14',
 'ENSG00000205542.11',
 'ENSG00000167658.16',
 'ENSG00000096384.20',
 'ENSG00000137154.13',
 'ENSG00000198034.11',
 'ENSG00000197746.14',
 'ENSG00000156508.19',
 'ENSG00000171345.13',
 'ENSG00000148303.17',
 'ENSG00000111640.15',
 'ENSG00000108821.14',
 'ENSG00000204287.14',
 'ENSG00000113140.11',
 'ENSG00000100219.16',
 'ENSG00000168542.16',
 'ENSG00000142541.18',
 'ENSG00000100316.16',
 'ENSG0000

In [13]:
top_genes_list = avg_expr.sort_values(ascending=False).head(1000)
top_genes_list

Ensembl_ID
ENSG00000198938.2     14.326857
ENSG00000198712.1     14.203714
ENSG00000198804.2     14.165769
ENSG00000198886.2     14.044224
ENSG00000210082.2     13.725247
                        ...    
ENSG00000065427.15     6.783693
ENSG00000121966.7      6.782731
ENSG00000108179.14     6.780712
ENSG00000181222.19     6.779358
ENSG00000143418.20     6.776228
Length: 1000, dtype: float64

In [16]:
top_gene_ids = set(g.split('.')[0] for g in top_genes)
print(top_gene_ids)

{'ENSG00000136636', 'ENSG00000119318', 'ENSG00000125817', 'ENSG00000181019', 'ENSG00000100994', 'ENSG00000008283', 'ENSG00000185236', 'ENSG00000113013', 'ENSG00000115524', 'ENSG00000131238', 'ENSG00000152583', 'ENSG00000115461', 'ENSG00000133321', 'ENSG00000065154', 'ENSG00000099622', 'ENSG00000078668', 'ENSG00000204628', 'ENSG00000188612', 'ENSG00000182326', 'ENSG00000133872', 'ENSG00000175567', 'ENSG00000198258', 'ENSG00000129009', 'ENSG00000100234', 'ENSG00000165283', 'ENSG00000226084', 'ENSG00000131236', 'ENSG00000167978', 'ENSG00000103363', 'ENSG00000011600', 'ENSG00000178741', 'ENSG00000092199', 'ENSG00000149932', 'ENSG00000114850', 'ENSG00000104852', 'ENSG00000106537', 'ENSG00000002549', 'ENSG00000196821', 'ENSG00000204388', 'ENSG00000130726', 'ENSG00000112306', 'ENSG00000160014', 'ENSG00000004478', 'ENSG00000158716', 'ENSG00000133112', 'ENSG00000169242', 'ENSG00000101335', 'ENSG00000213639', 'ENSG00000171858', 'ENSG00000166033', 'ENSG00000116030', 'ENSG00000198804', 'ENSG000001

In [7]:
from pathlib import Path
def normalize(filename):

    output_fa = f"{filename}_1000bp.fa"
    filename = filename + ".fa"

    TARGET_LEN = 1000

    def center_crop(seq, L):
        """Trim sequence to the center L bases."""
        if len(seq) == L:
            return seq
        if len(seq) < L:
            # Pad equally on both sides with N
            pad = L - len(seq)
            left = pad // 2
            right = pad - left
            return "N"*left + seq + "N"*right
        # longer seq → crop centered
        start = (len(seq) - L) // 2
        return seq[start : start+L]

    # Read in FASTA
    seqs = []
    headers = []
    with open(filename) as f:
        cur = []
        header = None
        for line in f:
            line = line.strip()
            if line.startswith(">"):
                if cur:
                    seqs.append("".join(cur).upper())
                    cur = []
                header = line
                headers.append(header)
            else:
                cur.append(line)
        if cur:
            seqs.append("".join(cur).upper())

    print("Loaded:", len(seqs))
    print("Original length stats:", min(len(s) for s in seqs), max(len(s) for s in seqs))

    # Normalize
    fixed = [center_crop(s, TARGET_LEN) for s in seqs]

    print("New length stats:", min(len(s) for s in fixed), max(len(s) for s in fixed))

    # Save new FASTA
    with open(output_fa, "w") as out:
        for h, s in zip(headers, fixed):
            out.write(h + "\n")
            for i in range(0, len(s), 80):
                out.write(s[i:i+80] + "\n")

    print("Saved normalized FASTA →", output_fa)


In [8]:
filename = "top1000_promoters_raw"
normalize(filename)

Loaded: 999
Original length stats: 848 1200
New length stats: 1000 1000
Saved normalized FASTA → top1000_promoters_raw_1000bp.fa


In [9]:
filename = "top10000_promoters_raw"
normalize(filename)

Loaded: 9993
Original length stats: 848 1200
New length stats: 1000 1000
Saved normalized FASTA → top10000_promoters_raw_1000bp.fa
